Quantum Leadership Training Program 2025

# Quantum Kernel estimation

Kifumi Numata, IBM Quantum    (Nov 24, 2025)</br>
© 2025 International Business Machines Corporation

### Table of contents:
1. Data encoding
2. Classification of hand-written image data
   - Exercise 1
3. Classification of fashion data
   - Exercise 2

In [ ]:
# Required packages for Google Colab
#!pip install qiskit qiskit-ibm-runtime qiskit-aer qiskit[visualization]
#!pip install -U scikit-learn

In [ ]:
import qiskit
qiskit.__version__

## 1. Data encoding

### 1.1 Basis encoding

Basis encoding encodes a classical $P$ -bit string into a computational basis state of a $P$ -qubit system. Once you have calculated the quantum state of encoding a data point, you can prepare it using Qiskit `initialize` function.

Let's encode $\vec{x} = (5,7,0)$. This is encoded as state $|x\rangle= \frac{1}{\sqrt{3}}(|101\rangle+|111\rangle+|000\rangle)$ because $x_1=101, x_2=111, x_3 = 000$ in binary notation.

In [ ]:
import math
desired_state = [
    1 / math.sqrt(3), # |000>
    0,
    0,
    0,
    0,
    1 / math.sqrt(3), # |101>
    0,
    1 / math.sqrt(3)] # |111>

desired_state

After you have converted the data into a quantum state, you can create this state with the `initialize` function. 

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3)
qc.initialize(desired_state, [0,1,2])
qc.decompose().decompose().decompose().decompose().decompose().draw(output = 'mpl')

Let's execute this circuit. The probability amplitude will be approximately the same.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import plot_distribution

qc.measure_all()
backend = AerSimulator()
sampler = Sampler(backend)
job = sampler.run([qc])

counts = job.result()[0].data.meas.get_counts()
plot_distribution(counts)    # show the probability distribution

### 1.2 Amplitude encoding

Amplitude encoding encodes data into the amplitudes of a quantum state.  
Let's encode $\vec{x} = (4,8,5)$ using amplitude encoding. First, compute the constant $\alpha$ to normalize the vector you seek:

$$\sum_{i=1}^N \left|x_i\right|^2 = 4^2+8^2+5^2 = 105 = \left|\alpha\right|^2 \rightarrow \alpha = \sqrt{105}$$

Using this constant, the quantum state is obtained by putting the data in order of the amplitude of the computational basis as follows:

$$|\psi(\vec{x})\rangle = \frac{1}{\sqrt{105}}(4|00\rangle+8|01\rangle+5|10\rangle+0|11\rangle)$$

In the example above, the number of elements of the vector $N=3$ is not a power of 2. If $N$ is not a power of 2, then the smallest $n$ that is $2^n\geq N$ is the number of qubits. Fill the remaining amplitude vectors with zero. This time, $n=2$ is the smallest $n$, so we prepared it in the state of 2 qubits.
 
As with basis encoding, once you have calculated the state of encoding your dataset, Qiskit can prepare it using the `initialize` function. 


In [ ]:
desired_state = [
    1 / math.sqrt(105) * 4,
    1 / math.sqrt(105) * 8,
    1 / math.sqrt(105) * 5,
    1 / math.sqrt(105) * 0]

qc = QuantumCircuit(2)
qc.initialize(desired_state, [0,1])

qc.decompose(reps=5).draw(output = 'mpl')

Let's measure this quantum circuit. You can see the data is reflected in the probability amplitude.

In [ ]:
qc.measure_all()
backend = AerSimulator()
sampler = Sampler(backend)
job = sampler.run([qc])

counts = job.result()[0].data.meas.get_counts()
plot_distribution(counts)   

### 1.3 Angle encoding

Angle encoding encodes data to the angle of a rotating gate, such as $R_X$ gate or $R_Y$ gate. Angule encoding is important in Quantum Machine Learning algorithms that use quantum feature maps, such as quantum-supported vector machines (QSVMs) and variational quantum circuits (VQCs). 

Below, we encode $\vec{x}= (0, \pi/4, \pi/2)$ with the $R_Y$ gate. 


In [ ]:
qc = QuantumCircuit(3)
qc.ry(0, 0)
qc.ry(2*math.pi/4, 1)
qc.ry(2*math.pi/2, 2)
qc.draw(output = 'mpl')

### 1.4 Enhanced angle encoding

Enhanced angle coding encodes data to the phase. It also includes multipile qubit gates for entanglement. Qiskit's [Circuit Library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library) has feature maps that enhanced angle encoding. 

#### Z feature map

[`z_feature_map`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.z_feature_map) (ZFM) is a simple feature map that is a natural extension of angle encoding. Below we will create a circuit of ZFM with three iterations with two-dimensional data.

In [ ]:
from qiskit.circuit.library import z_feature_map
from math import pi

zfeature_map = z_feature_map(feature_dimension=2, reps=3)
zfeature_map.draw("mpl")

Use `assign_parameters` to input data to the parameters. 

In [ ]:
zfeature_map = zfeature_map.assign_parameters([(1/2)*pi/2, (1/2)*pi/3])
zfeature_map.draw("mpl")

#### ZZ feature map
The [`zz_feature_map`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.zz_feature_map) (ZZFM) extends the ZFM with the inclusion of two-qubit entangling gates, specifically the $ZZ$-rotation gate,  $R_{ZZ}(\theta)$. The ZZFM is conjectured to be generally expensive to compute on a classical computer, unlike the ZFM.

The following is a four-dimensional data, with entanglement being a linear structure (`linear`) and repetition being a single ZZFM circuit.

In [ ]:
from qiskit.circuit.library import zz_feature_map
 
feature_dim = 4
zzfeature_map = zz_feature_map(
    feature_dimension=feature_dim, entanglement='linear', reps=1
)
zzfeature_map.draw("mpl", scale=1)

#### Pauli feature map

The [`pauli_feature_map`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.pauli_feature_map) (PFM) is the generalization of the ZFM and ZZFM to use arbitrary Pauli gates. The following PFM is three-dimensional data, the entanglement is `circular` which is linear and corresponding the qubit pairs at both ends, and the Pauli operator with "Y", "XX". 

In [ ]:
from qiskit.circuit.library import pauli_feature_map
 
feature_dim = 3
pfmap = pauli_feature_map(
    feature_dimension=feature_dim, entanglement='circular', reps=1, paulis=["Y", "XX"]
)
 
pfmap.draw("mpl", scale=1.5)

#### efficient_su2

You can also use a parameterized circuit as a feature map. The following is a common and useful example of encoding using entanglement, the [`efficient_su2`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.efficient_su2) circuit. This circuit can encode eight features with just two qubits, for example. 

In [ ]:
from qiskit.circuit.library import efficient_su2
 
circuit = efficient_su2(num_qubits=2, reps=1, insert_barriers=True)
circuit.draw(output="mpl")

## 2. Classification of hand-written image data
### Data preparation
We handle a subset of `0` and `1` from the handwritten digit image data set (MNIST data set).

In [ ]:
# Import Scikit-learn
from sklearn import datasets
from sklearn.model_selection import train_test_split 
from sklearn.svm import SVC 
from sklearn.decomposition import PCA # Principal component analysis
from sklearn.preprocessing import StandardScaler, MinMaxScaler 

In [ ]:
# load two classes of data (0 and 1) from a numeric data set
digits = datasets.load_digits(n_class=2)   

# Plot the first 10 images loaded
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 10, figsize=(15, 15), subplot_kw={'xticks':[], 'yticks':[]}, gridspec_kw=dict(hspace=0.5, wspace=0.5))
for i, ax in enumerate(axes.flat):
    ax.set_axis_off()
    ax.imshow(digits.images[i], cmap=plt.cm.gray_r, interpolation='nearest')
    ax.set_title(digits.target[i])

You may see `0` in below data array. 0 is white and 16 is black.

In [ ]:
print(digits.images[0])

This data set contains a total of 360 data points. Each data point is an 8 x 8 numbered image, in an array, with each element being an integer from 0 (white) to 16 (black). As with classical classification algorithms, the dataset must be split and normalized into training (25 samples) and testing (10 samples). To use this dataset for the quantum classification algorithm, the range is scaled between -1 and 1 and the dimension is reduced to the number of qubits used (4 in this case).

In [ ]:
import numpy as np
# Split the dataset
sample_train, sample_test, labels_train, labels_test = train_test_split(
     digits.data, digits.target, test_size=0.4, random_state=22)

# Dimension deletion
n_dim = 4
pca = PCA(n_components=n_dim).fit(sample_train)
sample_train = pca.transform(sample_train)
sample_test = pca.transform(sample_test)

# Normalization
std_scale = StandardScaler().fit(sample_train)
sample_train = std_scale.transform(sample_train)
sample_test = std_scale.transform(sample_test)

# Scaling
samples = np.append(sample_train, sample_test, axis=0)
minmax_scale = MinMaxScaler((-1, 1)).fit(samples)
sample_train = minmax_scale.transform(sample_train)
sample_test = minmax_scale.transform(sample_test)

# Sample 25 for training and 10 for testing 
train_size = 25
sample_train = sample_train[:train_size]
labels_train = labels_train[:train_size]

test_size = 10
sample_test = sample_test[:test_size]
labels_test = labels_test[:test_size]

In [ ]:
# Show the first data from training and testing
print(sample_train[0], labels_train[0])
print(sample_test[0], labels_test[0])

In [ ]:
# Show the label of training
print(labels_train)

### Encode data

We will encode this classical data into a quantum state using a quantum feature map (`zz_feature_map`). 

In [ ]:
# ZZFeatureMap with 4 features and 1 repetition 
zz_map = zz_feature_map(feature_dimension=4, entanglement='linear', reps=1, insert_barriers=True)
zz_map.draw('mpl')

### Quantum Kernel calculation - example

Using `zz_feature_map`, we create a quantum circuit that calculates the quantum kernel for the 0th and 1st pieces of training data. This value is also called the fidelity of the two quantum states.

In [ ]:
print(sample_train[0])
print(sample_train[1])

In [ ]:
zz_map = zz_feature_map(feature_dimension=4, reps=1, entanglement='linear')
qc_1 =zz_map.assign_parameters(sample_train[0])
qc_2 =zz_map.assign_parameters(sample_train[1])
fidelity_circuit = qc_1.copy()
fidelity_circuit.append(qc_2.inverse().decompose(), range(fidelity_circuit.num_qubits))
fidelity_circuit.measure_all()
fidelity_circuit.decompose().draw('mpl')

The parameter values of each rotation gate are a bit difficult to read, but you can see that this circuit is symmetrical. The left half is encoded with the training data sample_train[0], while the right half is encoded with the training data sample_train[1].

As an example, we measure the quantum kernel above and calculate the element of the kernel matrix (for the 0th and 1st data of the training data) as a percentage of the number of counts in the zero state.

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# Simulator
backend = AerSimulator()
# Transpile the circuit
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(fidelity_circuit)

In [ ]:
# Execute by Sampler
nshots = 1024
sampler = Sampler(backend)
job = sampler.run([isa_qc], shots=nshots)
result = job.result()

# Print the number of counting 
counts = result[0].data.meas.get_counts()
print(counts)

In [ ]:
# calculate the element of the kernel matrix
counts.get('0000')/sum(counts.values())

### Quantum Kernel Calculation for all elements

We repeat the above for each pair of training data to fill in the training kernel matrix.
We will use the same `zz_feature_map` to create a quantum circuit, and this time we will use `unitary_overlap` to calculate the fidelity, which is the element of the kernel matrix. 

In [ ]:
from qiskit.circuit.library import unitary_overlap
zz_map = zz_feature_map(feature_dimension=4, reps=1, entanglement='linear')
fidelity = unitary_overlap(zz_map, zz_map)
fidelity.measure_all()

Let's see what the circuit looks like.

In [ ]:
fidelity.draw('mpl', idle_wires=False, fold=-1)

To store the output, create an empty kernel matrix and an empty test kernel matrix.

In [ ]:
# empty kernel matrix
kernel_matrix = np.full((train_size, train_size), np.nan) # 25x25 for training kernel
test_matrix = np.full((test_size, train_size), np.nan) # 10x25 for test kernel

We will build the overlap circuits and calculate the kernel matrix elements. Since the kernel matrix is symmetrical to the diagonal, the number of calculations can be reduced by half by not repeating the same calculations. Also, if there is no noise, the diagonal element is 1, so this time we will not calculate the diagonal element and assign 1. Since we are calculating the (25×25×1/2-25) and (10×25) elements, the following cells may take a little longer to execute.

In [ ]:
%%time
num_shots = 1024

backend = AerSimulator()
sampler = Sampler(mode=backend)

for x1 in range(0,train_size):
    for x2 in range(x1+1,train_size):
        unitary1 = zz_map.assign_parameters(list(sample_train[x1]))
        unitary2 = zz_map.assign_parameters(list(sample_train[x2]))

        # Create the overlap circuit
        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()
    
        # These lines run the qiskit sampler primitive.
        counts = (sampler.run([overlap_circ], shots=num_shots).result()[0].data.meas.get_int_counts())

        # Assign the probability of the |0> state to the kernel matrix, 
        # and the transposed element (since this is an inner product)
        kernel_matrix[x1, x2] = counts.get(0, 0.0)/num_shots
        kernel_matrix[x2, x1] = counts.get(0, 0.0)/num_shots
    # Fill in on-diagonal elements with 1, again, since this is an inner-product corresponding to probability
    kernel_matrix[x1,x1] =1    

print("training done")

# Similar process to above, but for testing data.
for x1 in range(0,test_size):
    for x2 in range(0,train_size):
        unitary1 = zz_map.assign_parameters(list(sample_test[x1]))
        unitary2 = zz_map.assign_parameters(list(sample_train[x2]))

        # Create the overlap circuit
        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()
    
        counts = (sampler.run([overlap_circ], shots=num_shots).result()[0].data.meas.get_int_counts())

        test_matrix[x1, x2] = counts.get(0, 0.0)/num_shots

print("test matrix done")

Let's see the kernel_matrix and test_matrix.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.asmatrix(kernel_matrix),
              interpolation='nearest', origin='upper', cmap='Blues')
axs[0].set_title("kernel matrix")
axs[1].imshow(np.asmatrix(test_matrix),
              interpolation='nearest', origin='upper', cmap='Reds')
axs[1].set_title("test matrix")
plt.show()

### Learning with Classical Calculations

We use classical machine learning algorithms for training. Using Scikit-Learn's support vector classifier (SVC) `sklearn.svc`, we specify `kernel = precomputed` to use a pre-computed kernel. 

In [ ]:
from sklearn.svm import SVC # SVM Classification
#Specify that you want to use a pre-computed kernel matrix
qml_svc = SVC(kernel="precomputed")

We will use `SVC.fit` to enter the kernel matrix and training labels for training. `SVC.predict` uses test_matrix to predict the label of the test data. `SVC.score` scores the test data and returns accuracy.

In [ ]:
#Feed in the pre-computed matrix and the labels of the training data. The classical algorithm gives you a fit.
qml_svc.fit(kernel_matrix, labels_train)

#Predicts labels for test data using kernel matrix with training and test data
label_predict = qml_svc.predict(test_matrix)
print(label_predict, labels_test, sep="\n")

#Now use the .score to test your data, using the matrix of test data, and test labels as your inputs.
qml_score_precomputed_kernel = qml_svc.score(test_matrix, labels_test)
print(f"Precomputed kernel classification test score: {qml_score_precomputed_kernel}")

We have created the kernel classification system of 90% accuracy.

## Exercise 1

Use [`pauli_feature_map`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.pauli_feature_map) to create your own feature map and create kernel matrix for the same dataset to improve learning accuracy.

In [ ]:
feature_dim = ##your code goes here##
p_map = pauli_feature_map(
    ##your code goes here##
)
p_map.draw("mpl", scale=1.5)

In [ ]:
# empty kernel matrix
kernel_matrix_ex1 = np.full((train_size, train_size), np.nan) # 25x25 for training kernel
test_matrix_ex1 = np.full((test_size, train_size), np.nan) # 10x25 for test kernel

In [ ]:
%%time
num_shots = 1024

backend = AerSimulator()
sampler = Sampler(mode=backend)

for x1 in range(0,train_size):
    for x2 in range(x1+1,train_size):
        unitary1 = p_map.assign_parameters(list(sample_train[x1]))
        unitary2 = p_map.assign_parameters(list(sample_train[x2]))

        # Create the overlap circuit
        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()
    
        # These lines run the qiskit sampler primitive.
        counts = (sampler.run([overlap_circ], shots=num_shots).result()[0].data.meas.get_int_counts())

        # Assign the probability of the |0> state to the kernel matrix, 
        # and the transposed element (since this is an inner product)
        kernel_matrix_ex1[x1, x2] = counts.get(0, 0.0)/num_shots
        kernel_matrix_ex1[x2, x1] = counts.get(0, 0.0)/num_shots
    # Fill in on-diagonal elements with 1, again, since this is an inner-product corresponding to probability
    kernel_matrix_ex1[x1,x1] =1    

print("training done")

# Similar process to above, but for testing data.
for x1 in range(0,test_size):
    for x2 in range(0,train_size):
        unitary1 = p_map.assign_parameters(list(sample_test[x1]))
        unitary2 = p_map.assign_parameters(list(sample_train[x2]))

        # Create the overlap circuit
        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()
    
        counts = (sampler.run([overlap_circ], shots=num_shots).result()[0].data.meas.get_int_counts())

        test_matrix_ex1[x1, x2] = counts.get(0, 0.0)/num_shots

print("test matrix done")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.asmatrix(kernel_matrix_ex1),
              interpolation='nearest', origin='upper', cmap='Blues')
axs[0].set_title("kernel matrix_ex1")
axs[1].imshow(np.asmatrix(test_matrix_ex1),
              interpolation='nearest', origin='upper', cmap='Reds')
axs[1].set_title("test matrix_ex1")
plt.show()

In [ ]:
#Specify that you want to use a pre-computed kernel matrix
qml_svc = SVC(kernel="precomputed")

#Feed in the pre-computed matrix and the labels of the training data. The classical algorithm gives you a fit.
qml_svc.fit(kernel_matrix_ex1, labels_train)

#Predicts labels for test data using kernel matrix with training and test data
label_predict_ex1 = qml_svc.predict(test_matrix_ex1)
print(label_predict_ex1, labels_test, sep="\n")

#Now use the .score to test your data, using the matrix of test data, and test labels as your inputs.
qml_score_precomputed_kernel = qml_svc.score(test_matrix_ex1, labels_test)
print(f"Precomputed kernel classification test score: {qml_score_precomputed_kernel}")

## 3. Classification of fashion data

The data covered here is a subset of the clothing image dataset called [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist/tree/master), which is a variant of the MNIST dataset. 

![Fashion-MNIST](https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/refs/heads/master/doc/img/fashion-mnist-sprite.png)

We will classify the images with the following labels:

- Label 2: Pullover
- Label 3: Dress

First, we will load the dataset and display one image for each class.


In [ ]:
# Load the data
DATA_PATH = 'fashion.npz'
data = np.load(DATA_PATH)

sample_train = data['sample_train']
labels_train = data['labels_train']
sample_test = data['sample_test']

# Split the dataset
sample_train, sample_test, labels_train, labels_test = train_test_split(
    sample_train, labels_train, test_size=0.2, random_state=42)

# Show the one image for each class
fig = plt.figure()
LABELS = [2,3]
num_labels = len(LABELS)
for i in range(num_labels):
    ax = fig.add_subplot(2, num_labels, i+1)
    img = sample_train[labels_train==LABELS[i]][0].reshape((28, 28))
    ax.imshow(img, cmap="Greys")

Next, we preprocess the datasets:

- Dimensional compression by principal component analysis (PCA)
- Normalization
- Scaling
- Samples 50 for training and 10 for testing

In [ ]:
# Dimension deletion
N_DIM = 5
pca = PCA(n_components=N_DIM).fit(sample_train)
sample_train = pca.transform(sample_train)
sample_test = pca.transform(sample_test)

# Normalization
std_scale = StandardScaler().fit(sample_train)
sample_train = std_scale.transform(sample_train)
sample_test = std_scale.transform(sample_test)

# Scaling
samples = np.append(sample_train, sample_test, axis=0)
minmax_scale = MinMaxScaler((-1, 1)).fit(samples)
sample_train = minmax_scale.transform(sample_train)
sample_test = minmax_scale.transform(sample_test)

# Samples 50 for training and 10 for testing
train_size = 50
sample_train = sample_train[:train_size]
labels_train = labels_train[:train_size]

test_size = 10
sample_test = sample_test[:test_size]
labels_test = labels_test[:test_size]

In [ ]:
# Show the first data from training and testing
print(sample_train[0], labels_train[0])
print(sample_test[0], labels_test[0])

## Exercise 2

Build your own kernel matrix for this fashion date and learn it using the classical SVM. Check the learning accuracy.

In [ ]:
fmap_ex2 =

fmap_ex2.decompose().draw('mpl', idle_wires=False, fold=-1)

In [ ]:
# empty kernel matrix
kernel_matrix_ex2 = np.full((train_size, train_size), np.nan) # 50x50 for training kernel
test_matrix_ex2 = np.full((test_size, train_size), np.nan) # 10x50 for test kernel

In [ ]:
%%time
num_shots = 1024

backend = AerSimulator()
sampler = Sampler(mode=backend)

for x1 in range(0,train_size):
    for x2 in range(x1+1,train_size):
        unitary1 = ##your code goes here##
        unitary2 = ##your code goes here##

        # Create the overlap circuit
        overlap_circ = ##your code goes here##
        overlap_circ.measure_all()
    
        # These lines run the qiskit sampler primitive.
        counts = (
            ##your code goes here##
        )

        # Assign the probability of the |0> state to the kernel matrix, 
        # and the transposed element (since this is an inner product)
        kernel_matrix_ex2[x1, x2] = ##your code goes here##
        kernel_matrix_ex2[x2, x1] = ##your code goes here##
    # Fill in on-diagonal elements with 1, again, since this is an inner-product corresponding to probability
    kernel_matrix_ex2[x1,x1] = ##your code goes here##   

print("training done")

# Similar process to above, but for testing data.
for x1 in range(0,test_size):
    for x2 in range(0,train_size):
        unitary1 = ##your code goes here##
        unitary2 = ##your code goes here##

        # Create the overlap circuit
        overlap_circ = ##your code goes here##
        overlap_circ.measure_all()
    
        counts = (
            ##your code goes here##
        )

        test_matrix_ex2[x1, x2] = ##your code goes here##

print("test matrix done")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.asmatrix(kernel_matrix_ex2),
              interpolation='nearest', origin='upper', cmap='Blues')
axs[0].set_title("kernel matrix_ex2")
axs[1].imshow(np.asmatrix(test_matrix_ex2),
              interpolation='nearest', origin='upper', cmap='Reds')
axs[1].set_title("test matrix_ex2")
plt.show()

In [ ]:
#Specify that you want to use a pre-computed kernel matrix
qml_svc = SVC(kernel="precomputed")

#Feed in the pre-computed matrix and the labels of the training data. The classical algorithm gives you a fit.
qml_svc.fit(kernel_matrix_ex2, labels_train)

#Predicts labels for test data using kernel matrix with training and test data
label_predict = qml_svc.predict(test_matrix_ex2)
print(label_predict, labels_test, sep="\n")

#Now use the .score to test your data, using the matrix of test data, and test labels as your inputs.
qml_score_precomputed_kernel = qml_svc.score(test_matrix_ex2, labels_test)
print(f"Precomputed kernel classification test score: {qml_score_precomputed_kernel}")

## Bonus Exercise

Find whether the dataset `unknown_data` is pullover (label 2) or dress (label 3) using the quantum kernel matrix and the SVM. `unknown_data` is a dataset of 10 identical labels that have been dimension-deleted, normalized, and scaled.

In [ ]:
unknown_data = [
    [0.31559114, -0.22212598, -0.01367075, 0.26984272, 0.09538095],
    [0.61950235, 0.11437916, -0.71187257, -0.20481753, 0.33026303],
    [0.81223033, 0.16992666, 0.01845638, 0.03393138, 0.05011245],
    [0.84295123, 0.07871908, -0.73484067, 0.05868912, 0.11878952],
    [0.88399294, 0.05947462, -0.44905742, -0.41026581, -0.19594553],
    [0.77159032, 0.19355832, 0.49900056, -1.0, -0.09096834],
    [0.06948713, -0.5180201, 0.45473245, -0.02749577, -0.38410395],
    [1.0, 0.20637581, -0.24059816, 0.24122204, 0.28130522],
    [-0.13116065, -0.06375504, -0.1368573, 0.8475517, -0.01036726],
    [-0.37941815, -0.76067967, -0.22019796, 0.17362436, -0.23302668]
]